# 01 数据工程基础：获取、清洗、划分

> 前置：`03-probability`（统计量）、`01-linear-algebra`（向量与矩阵）。
> 目标：把"真实数据"变成"模型能吃的东西"——这是所有实际问题里**最花时间的一步**（业内常说 80/20：数据八分、建模二分）。

## 为什么数据工程是主战场

Kaggle 竞赛与工业项目的经验一致：**模型选型只决定上限的 20%，数据质量决定剩下的 80%**。本课建立数据工程的四个环节闭环：

1. **获取**：真实数据从哪来（Kaggle、公开数据集、业务数据库）；
2. **清洗**：缺失值、类型错误、离群点、重复样本；
3. **防泄漏**：目标信息绝不能混进特征；
4. **划分**：训练/验证/测试怎么切（分层、时序），决定了评估是否可信。

本课用 Kaggle 经典竞赛 **Titanic（泰坦尼克号生存预测）** 的真实数据演示（公开 CDN 镜像，无需 Kaggle 账号；数据不可用时会给出明确指引，保证可复现）。

In [1]:
# 本模块通用导入（全部 CPU 即可运行）
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

# 中文字体兼容（Windows / macOS）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

numpy 2.5.2 | pandas 3.0.5 | sklearn 1.9.0


In [2]:
import io, urllib.request

TITANIC_URLS = [
    "https://cdn.jsdelivr.net/gh/datasciencedojo/datasets@master/titanic.csv",
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
]

def load_titanic():
    """Titanic 数据：本地缓存优先（模块根目录），CDN 在线下载兜底（各 15s 超时）。"""
    for p in (os.path.join("..", "titanic.csv"), "titanic.csv"):
        if os.path.exists(p):
            df = pd.read_csv(p)
            print("已加载本地缓存 titanic.csv：", df.shape)
            return df
    for url in TITANIC_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15) as r:
                data = r.read()
            with open(os.path.join("..", "titanic.csv"), "wb") as f:
                f.write(data)
            df = pd.read_csv(io.BytesIO(data))
            print("已在线下载真实 Titanic 数据：", df.shape)
            return df
        except Exception:
            continue
    raise RuntimeError(
        "Titanic 数据不可用：无本地缓存且在线下载失败。"
        "请联网后重跑本 cell，或手动把 titanic.csv 放到模块根目录 08-applied-ml/ 下。")

df = load_titanic()
print("行数 =", len(df), " 列数 =", len(df.columns))
print(df.head(3).to_string())
print("\n数据类型分布：")
print(df.dtypes.value_counts())
print("\n数值列统计：")
print(df.describe().T[["count", "mean", "min", "max"]].round(3))

已加载本地缓存 titanic.csv： (891, 12)
行数 = 891  列数 = 12
   PassengerId  Survived  Pclass                                                 Name     Sex   Age  SibSp  Parch            Ticket     Fare Cabin Embarked
0            1         0       3                              Braund, Mr. Owen Harris    male  22.0      1      0         A/5 21171   7.2500   NaN        S
1            2         1       1  Cumings, Mrs. John Bradley (Florence Briggs Thayer)  female  38.0      1      0          PC 17599  71.2833   C85        C
2            3         1       3                               Heikkinen, Miss. Laina  female  26.0      0      0  STON/O2. 3101282   7.9250   NaN        S

数据类型分布：
int64      5
str        5
float64    2
Name: count, dtype: int64

数值列统计：
             count     mean   min      max
PassengerId  891.0  446.000  1.00  891.000
Survived     891.0    0.384  0.00    1.000
Pclass       891.0    2.309  1.00    3.000
Age          714.0   29.699  0.42   80.000
SibSp        891.0    0.523  0

## 数据质量四问

拿到数据先问四件事：

| 问题 | 检查方法 | Titanic 实例 |
|------|---------|--------------|
| 缺失？ | `isna().sum()` | Age、Cabin、Embarked 有缺失 |
| 类型错？ | `dtypes` | Sex/Embarked 是文本，需编码 |
| 离群？ | `describe()` 看 min/max | Fare 有 512 的极端值 |
| 冗余？ | `duplicated()` | 重复样本 |

**缺失值处理原则**：先看缺失率与缺失机制，再选策略——中位数/众数填充（简单稳健）、模型预测填充（更精细）、或转成"是否缺失"特征（缺失本身可能是信息）。

In [3]:
print("各列缺失值：")
print(df.isna().sum()[df.isna().sum() > 0])

# 数值列 Age：中位数填充（对偏态分布比均值稳健）
df["Age"] = df["Age"].fillna(df["Age"].median())
# 类别列 Embarked：众数填充
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin 缺失率 77% —— 直接丢弃列，改为"是否有 Cabin"二值特征（缺失本身是信号）
df["HasCabin"] = df["Cabin"].notna().astype(int)
print("\nCabin 缺失率 = %.1f%%  → 转为 HasCabin 特征：" % (1 - df["HasCabin"].mean()))
print(df["HasCabin"].value_counts())

# 离群点：看 Fare 极端值（> 300 的豪华舱）
print("\nFare 极端值（>300）：", int((df["Fare"] > 300).sum()), "个")
print("离群点处理策略：保留但记录（真实数据里豪华舱是真实存在的人群，不应随手删除）")

各列缺失值：
Age         177
Cabin       687
Embarked      2
dtype: int64

Cabin 缺失率 = 0.8%  → 转为 HasCabin 特征：
HasCabin
0    687
1    204
Name: count, dtype: int64

Fare 极端值（>300）： 3 个
离群点处理策略：保留但记录（真实数据里豪华舱是真实存在的人群，不应随手删除）


## 标签泄漏：最隐蔽的坑

**泄漏 = 训练时用到了"只有测试时才知道的信息"**。比如：

- 把测试集的统计量（如全局均值）用于训练数据填充 → 泄漏；
- 特征里包含目标本身的衍生物（如用"是否退款"预测"是否欺诈"）→ 泄漏；
- 时序问题里用未来数据做特征 → 泄漏。

Titanic 里的安全做法：`Survived` 只作标签，绝不进特征；所有填充统计量只在**训练集**上计算（严格做法），本课为演示先全量填充，08 课会做严格版。

**特征与标签分离**是第一步。

In [4]:
# 特征与标签分离
target = "Survived"
feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "HasCabin"]
X = df[feature_cols].copy()
y = df[target].astype(int)
print("特征维度：", X.shape, "  标签分布：", y.value_counts().to_dict())

# 类别编码：文本列 → 数值（one-hot，drop_first 避免共线性）
X = pd.get_dummies(X, columns=["Sex", "Embarked"], drop_first=True)
print("编码后特征：", list(X.columns))

from sklearn.model_selection import train_test_split
# 普通随机划分
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
# 分层划分：保证两边类别比例一致
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("\n普通划分  训练集正类比例 = %.3f  测试集 = %.3f" % (y_tr.mean(), y_te.mean()))
print("分层划分  训练集正类比例 = %.3f  测试集 = %.3f" % (y_tr2.mean(), y_te2.mean()))

# 时序场景：时间序列绝不能随机 shuffle（未来信息泄漏）
rng = np.random.default_rng(0)
t = np.arange(200)
ts = pd.DataFrame({"time": t, "value": np.sin(t / 10) + rng.normal(0, 0.1, 200)})
print("\n时序切分：训练 = 前 80% 时间，验证 = 后 20% —— 绝不混入未来数据")

特征维度： (891, 8)   标签分布： {0: 549, 1: 342}
编码后特征： ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'Sex_male', 'Embarked_Q', 'Embarked_S']

普通划分  训练集正类比例 = 0.376  测试集 = 0.413
分层划分  训练集正类比例 = 0.383  测试集 = 0.385

时序切分：训练 = 前 80% 时间，验证 = 后 20% —— 绝不混入未来数据


## 类别不平衡

当正负样本比例悬殊（如欺诈 0.1%、罕见病 1%），**准确率会骗人**（全猜负类也有 99.9%）。

处理三板斧（本模块 02/05 课会用上）：
1. **指标换掉**：不用 accuracy，改用 PR-AUC / F1（05 课细讲）；
2. **样本加权**：`class_weight="balanced"` 或重采样；
3. **别动测试集**：测试集必须保持真实分布。

先记住原则：**不平衡问题的敌人是"指标被多数类绑架"**。

In [5]:
# 保存本次划分索引（供后续课程复用）
np.save("split_idx.npy", {"train": X_tr2.index.values, "test": X_te2.index.values})
print("已保存 train/test 索引：", len(X_tr2), "/", len(X_te2))

# 数据工程闭环小结
print("\n数据工程闭环：")
print("  获取真实数据 → 缺失值处理 → 类型/编码 → 特征标签分离 → 分层划分")

已保存 train/test 索引： 712 / 179

数据工程闭环：
  获取真实数据 → 缺失值处理 → 类型/编码 → 特征标签分离 → 分层划分


## 课后练习（Kaggle：Titanic）

竞赛链接：<https://www.kaggle.com/c/titanic>

1. **数据探查**：打开 Titanic 训练集，回答——每列缺失率多少？`Sex` 对生存率的影响（分组算 `Survived` 均值）？`Pclass` 呢？
2. **编码练习**：把 `Cabin` 的首字母（A/B/C/D/E）提出来做特征，看是否有区分度——"特征工程从读懂字段含义开始"。
3. **思考题**：如果测试集只有 418 行而训练集 891 行，为什么不能用测试集的分布做填充统计？如果用了，评估会怎样失真？
4. **进阶**：在 Kaggle 注册账号，进入 Titanic 竞赛页面，阅读 Data 页的字段说明（data dictionary），对照本课的数据质量四问逐列检查。